# 04 — Results Analysis & Visualization

Load benchmark results from `results/tables/all_results.csv` and generate
comparison tables, charts, and key findings.

**Run this on**: Any machine — no GPU needed.

### Expected rows in the CSV
| Model name | Description |
|---|---|
| `gemma4-e2b-baseline` | Unmodified IT model, 4-bit (notebook 01) |
| `gemma4-e2b-medical-finetuned` | After UnSloth QLoRA fine-tuning, 4-bit (notebook 02) |
| `gemma4-e2b-med-gptq-8bit` | GPTQ 8-bit post-quantization |
| `gemma4-e2b-med-gptq-4bit` | GPTQ 4-bit post-quantization |
| `gemma4-e2b-med-gptq-3bit` | GPTQ 3-bit post-quantization |
| `gemma4-e2b-med-awq-4bit` | AWQ 4-bit post-quantization |
| `gemma4-e2b-med-bnb-nf4` | BitsAndBytes NF4 (4-bit) |
| `gemma4-e2b-med-bnb-int8` | BitsAndBytes INT8 (8-bit) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os

matplotlib.rcParams["figure.figsize"] = (12, 6)
matplotlib.rcParams["font.size"] = 11

os.makedirs("results/figures", exist_ok=True)

df = pd.read_csv("results/tables/all_results.csv")
print(f"Rows loaded: {len(df)}")
df

## 1. Comparison Table

In [ ]:
display_cols = [
    "model", "perplexity_wikitext", "perplexity_medical",
    "pubmedqa_accuracy", "tokens_per_sec", "peak_vram_gb"
]
table = df[display_cols].copy()
table.columns = ["Model", "PPL (Wiki)", "PPL (Med)", "PubMedQA Acc", "Tok/s", "VRAM (GB)"]

# Calculate % deltas relative to the fine-tuned (pre-quantization) baseline
ft_rows = table[table["Model"] == "gemma4-e2b-medical-finetuned"]
if len(ft_rows) > 0:
    fp16_ppl  = ft_rows["PPL (Wiki)"].values[0]
    fp16_acc  = ft_rows["PubMedQA Acc"].values[0]
    fp16_vram = ft_rows["VRAM (GB)"].values[0]
    table["PPL Delta"]    = ((table["PPL (Wiki)"] - fp16_ppl) / fp16_ppl * 100).round(1).astype(str) + "%"
    table["Acc Delta"]    = ((table["PubMedQA Acc"] - fp16_acc) / fp16_acc * 100).round(1).astype(str) + "%"
    table["VRAM Savings"] = ((fp16_vram - table["VRAM (GB)"]) / fp16_vram * 100).round(1).astype(str) + "%"

print(table.to_markdown(index=False))
table

## 2. Accuracy vs Memory Tradeoff

In [ ]:
# Colour by quantization family
def model_color(name):
    if "gptq" in name:   return "#e74c3c"
    if "awq"  in name:   return "#3498db"
    if "bnb"  in name:   return "#2ecc71"
    return "#333333"   # baseline / fine-tuned

fig, ax = plt.subplots(figsize=(11, 7))

for _, row in df.iterrows():
    name = row["model"]
    ax.scatter(
        row["peak_vram_gb"], row["pubmedqa_accuracy"],
        s=130, c=model_color(name), zorder=5,
    )
    label = name.replace("gemma4-e2b-", "").replace("med-", "")
    ax.annotate(
        label, (row["peak_vram_gb"], row["pubmedqa_accuracy"]),
        textcoords="offset points", xytext=(8, 4), fontsize=9,
    )

# Legend
for label, color in [("GPTQ", "#e74c3c"), ("AWQ", "#3498db"),
                      ("BnB", "#2ecc71"), ("Baseline/FT", "#333333")]:
    ax.scatter([], [], c=color, label=label, s=100)
ax.legend(loc="lower right", fontsize=10)

ax.set_xlabel("Peak VRAM (GB)")
ax.set_ylabel("PubMedQA Accuracy")
ax.set_title("Accuracy vs Memory: Quantized Gemma 4 E2B Medical Model")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/accuracy_vs_memory.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Perplexity Comparison (WikiText-2 and Medical)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

short_names = df["model"].str.replace("gemma4-e2b-", "", regex=False)

axes[0].barh(short_names, df["perplexity_wikitext"], color="steelblue")
axes[0].set_xlabel("Perplexity")
axes[0].set_title("WikiText-2 Perplexity (lower = better)")
axes[0].invert_yaxis()

axes[1].barh(short_names, df["perplexity_medical"], color="coral")
axes[1].set_xlabel("Perplexity")
axes[1].set_title("Medical Text Perplexity (lower = better)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("results/figures/perplexity_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Inference Speed Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(short_names, df["tokens_per_sec"], color="mediumseagreen")
ax.set_xlabel("Tokens per Second")
ax.set_title("Inference Throughput — Gemma 4 E2B (higher = better)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("results/figures/speed_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Radar Chart — Accuracy · Speed · Memory

In [ ]:
import numpy as np

# Only show quantized variants (exclude baseline and FT rows)
quant_df = df[df["model"].str.contains("gptq|awq|bnb", regex=True)].reset_index(drop=True)

if len(quant_df) > 0:
    # Normalise each metric to [0, 1]
    acc_norm   = (quant_df["pubmedqa_accuracy"] - quant_df["pubmedqa_accuracy"].min()) / \
                 (quant_df["pubmedqa_accuracy"].max() - quant_df["pubmedqa_accuracy"].min() + 1e-9)
    speed_norm = (quant_df["tokens_per_sec"] - quant_df["tokens_per_sec"].min()) / \
                 (quant_df["tokens_per_sec"].max() - quant_df["tokens_per_sec"].min() + 1e-9)
    # Lower VRAM is better, so invert
    vram_norm  = 1 - (quant_df["peak_vram_gb"] - quant_df["peak_vram_gb"].min()) / \
                     (quant_df["peak_vram_gb"].max() - quant_df["peak_vram_gb"].min() + 1e-9)

    categories = ["Accuracy", "Speed", "Memory Eff."]
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    colors = plt.cm.Set2(np.linspace(0, 1, len(quant_df)))

    for i, row in quant_df.iterrows():
        values = [acc_norm.iloc[i], speed_norm.iloc[i], vram_norm.iloc[i]]
        values += values[:1]
        label = row["model"].replace("gemma4-e2b-med-", "")
        ax.plot(angles, values, linewidth=2, color=colors[i], label=label)
        ax.fill(angles, values, alpha=0.1, color=colors[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_title("Quantization Method Comparison (normalised)", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    plt.savefig("results/figures/radar_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No quantized variants found — run notebook 03 first.")

## 6. Key Findings Summary

In [ ]:
quant_df_all = df[df["model"].str.contains("gptq|awq|bnb", regex=True)]

if len(quant_df_all) == 0:
    print("No quantized variants in results yet. Run notebook 03 first.")
else:
    best_acc   = quant_df_all.loc[quant_df_all["pubmedqa_accuracy"].idxmax()]
    best_speed = quant_df_all.loc[quant_df_all["tokens_per_sec"].idxmax()]
    best_mem   = quant_df_all.loc[quant_df_all["peak_vram_gb"].idxmin()]
    best_ppl   = quant_df_all.loc[quant_df_all["perplexity_wikitext"].idxmin()]

    ft_rows = df[df["model"] == "gemma4-e2b-medical-finetuned"]

    print("KEY FINDINGS")
    print("=" * 65)
    print(f"Best accuracy retention : {best_acc['model']}")
    print(f"  PubMedQA acc          : {best_acc['pubmedqa_accuracy']:.4f}")
    print()
    print(f"Best throughput         : {best_speed['model']}")
    print(f"  Tokens/sec            : {best_speed['tokens_per_sec']:.1f}")
    print()
    print(f"Lowest VRAM             : {best_mem['model']}")
    print(f"  Peak VRAM             : {best_mem['peak_vram_gb']:.2f} GB")
    print()
    print(f"Best perplexity         : {best_ppl['model']}")
    print(f"  PPL (WikiText-2)      : {best_ppl['perplexity_wikitext']:.2f}")

    if len(ft_rows) > 0:
        ft = ft_rows.iloc[0]
        print()
        print(f"Fine-tuned baseline VRAM    : {ft['peak_vram_gb']:.2f} GB")
        savings = (ft["peak_vram_gb"] - best_mem["peak_vram_gb"]) / ft["peak_vram_gb"] * 100
        print(f"Best quantized VRAM savings : {savings:.1f}%")
        acc_drop = (
            (ft["pubmedqa_accuracy"] - best_acc["pubmedqa_accuracy"])
            / ft["pubmedqa_accuracy"] * 100
        )
        print(f"Accuracy drop (best quant)  : {acc_drop:.1f}%")

    # Baseline vs Fine-tuned comparison
    base_rows = df[df["model"] == "gemma4-e2b-baseline"]
    if len(base_rows) > 0 and len(ft_rows) > 0:
        base = base_rows.iloc[0]
        ft   = ft_rows.iloc[0]
        print()
        print("FINE-TUNING IMPACT (baseline -> fine-tuned)")
        print("-" * 65)
        print(f"  PubMedQA acc: {base['pubmedqa_accuracy']:.4f} -> {ft['pubmedqa_accuracy']:.4f}"
              f" ({(ft['pubmedqa_accuracy'] - base['pubmedqa_accuracy']) / base['pubmedqa_accuracy'] * 100:+.1f}%)")
        print(f"  PPL Wiki    : {base['perplexity_wikitext']:.2f} -> {ft['perplexity_wikitext']:.2f}")
        print(f"  PPL Medical : {base['perplexity_medical']:.2f} -> {ft['perplexity_medical']:.2f}")